In [1]:
# Libraries
import pandas as pd
import geopandas as gpd
import numpy as np 
import matplotlib.pyplot as plt
import rasterio
from rasterio.plot import show # Displaying rasters
from rasterio.warp import reproject, Resampling, calculate_default_transform # Reprojection
from rasterio import features # Rasterizing
from rasterio.enums import MergeAlg # Rasterizing while retaining attributes
np.set_printoptions(suppress = True) # Turn off scientific notation
from rio_cogeo.cogeo import cog_translate
from rio_cogeo.profiles import cog_profiles
from rio_cogeo import cog_validate, cog_info

In [13]:
# Only run initially!!

#from zipfile import ZipFile

#watershed_impairment_zip = './data/watershed_impairment/RPS_HUC12s_indicator.zip'
#watershed_impairment_out = './data/watershed_impairment'

#with ZipFile(watershed_impairment_zip, 'r') as zObject:
    # Extract downloaded watershed impairment data and store in data > watershed_impairment folder
#    zObject.extractall(path = watershed_impairment_out)

In [2]:
# Open watershed impairment shapefile

watershed_impairment_path = './data/watershed_impairment/RPS_HUC12s_indicator.shp'

impairment = gpd.read_file(watershed_impairment_path)[['HUC12_TEXT', 'pct_imp', 'geometry']]

print(f'Rows: {impairment.shape[0]}') # Shape
print(f'CRS: {impairment.crs.name}') # CRS

impairment.head(2)

Rows: 82915
CRS: USA_Contiguous_Albers_Equal_Area_Conic_USGS_version


,HUC12_TEXT,pct_imp,geometry
0,010100020101,0.0,"POLYGON ((2041707.821 2888147.626, 2041762.536..."
1,010100020102,0.0,"POLYGON ((2009760.815 2887170.984, 2009777.409..."


In [3]:
# I did multiple sanity checks to ensure that pct_imp values match CATAREA_IMP_PCT values in the og shapefile!!

In [4]:
# Reproject pop to NAD 83 CONUS Albers (epsg:5070)
impairment = impairment.to_crs('epsg:5070')
print(f'CRS: {impairment.crs}')

CRS: epsg:5070


In [5]:
# Check for null values - none!
impairment.isnull().values.any()

False

In [6]:
# Print min and max % impairment values

# ssp2 rcp45 2020
print(f"Min: {impairment['pct_imp'].min()}")
print(f"Max: {impairment['pct_imp'].max()}")

Min: 0.0
Max: 100.0


In [7]:
# Rasterize percent impairment values matching dist to gwt raster - DONE
dist_gwt_REF = './data/SSURGO_raw/dist_GWT/gwt_inches.tif'
impairment_raster = './data/watershed_impairment/impairment_raw_rasterized.tif'

geom_value = list(zip(impairment.geometry, impairment['pct_imp']))

with rasterio.open(dist_gwt_REF) as ref:
    # Copy profile
    profile = ref.profile.copy()
    
    # Update profile
    profile.update(compress = 'lzw')
        
    # Rasterize 
    rasterized = features.rasterize(
        geom_value,
        out_shape = ref.shape,
        transform = ref.transform,
        fill = -10, # Background fill
        all_touched = True,
        merge_alg = MergeAlg.replace,
        dtype = rasterio.float32
    )
    
    with rasterio.open(impairment_raster, 'w', **profile) as dst:
        dst.write(rasterized, 1)


In [8]:
# Verify rasterization (WITH MASKING)

impairment_raster = './data/watershed_impairment/impairment_raw_rasterized.tif'

with rasterio.open(impairment_raster, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [3]:
# Remove "excess" cells 
hsg_final_composite = './data/SSURGO_raw/hsg/hsg_FINAL_composite.tif' 
impairment_raster = './data/watershed_impairment/impairment_raw_rasterized.tif'
impairment_MATCH = './data/watershed_impairment/impairment_MATCH.tif'

# Open hsg composite raster - want conus cells to match THIS raster
with rasterio.open(hsg_final_composite) as conus:

    # Nodata value for the conus raster
    conus_nodata = conus.nodata 
    # Profile conus raster
    profile = conus.profile.copy()
    profile.update(dtype = rasterio.float32)

    # Open raster - want to convert any cells containing data where conus contains NODATA to nodata
    with rasterio.open(impairment_raster) as src:
        
        src_nodata = src.nodata # Nodata value 
        
        # Open output raster
        with rasterio.open(impairment_MATCH, 'w', **profile) as dst:
        
            for ji, window in conus.block_windows(1):
            
                # hsg composite raster data
                conus_data = conus.read(1, window = window)
            
                # Land cover raster data
                src_data = src.read(1, window = window)
            
                # Identify cells where conus_data == nodata value
                remove_mask = (conus_data == conus_nodata)
            
                # Convert cells in src where conus is nodata to the nodata value
                src_data[remove_mask] = src_nodata
            
                # Write out
                dst.write(src_data, 1, window = window)

In [4]:
# Verify match raster (WITH MASKING)

impairment_MATCH = './data/watershed_impairment/impairment_MATCH.tif'

with rasterio.open(impairment_MATCH, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [4]:
# Apply min-max scaling  

impairment_MATCH = './data/watershed_impairment/impairment_MATCH.tif'
impairment_standardized = './data/watershed_impairment/impairment_standardized.tif'

with rasterio.open(impairment_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(impairment_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)

Global min: 0.0
Global max: 100.0


In [5]:
# Verify min-max scaling

impairment_standardized = './data/watershed_impairment/impairment_standardized.tif'

with rasterio.open(impairment_standardized, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 153996, 'height': 96751, 'count': 1, 'crs': CRS.from_wkt('PROJCS["NAD83 / Conus Albers",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4269"]],PROJECTION["Albers_Conic_Equal_Area"],PARAMETER["latitude_of_center",23],PARAMETER["longitude_of_center",-96],PARAMETER["standard_parallel_1",29.5],PARAMETER["standard_parallel_2",45.5],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","5070"]]'), 'transform': Affine(30.0, 0.0, -2356125.0,
       0.0, -30.0, 3172575.0), 'blockxsize': 128, 'blockysize': 128, 'tiled': True, 'compress': 'lzw', 'interleave': 'band'}
Nodata: -10.0
CRS: EPSG:5070
Resolu

In [11]:
# Save out preprocessed dist GWT as cloud optimized GeoTiff - save to scratch 

impairment_raster = './data/watershed_impairment/impairment_raw_rasterized.tif'
cog_out_impairment_raw_viewing = '/scratch/bfqp/cchan2/rasters/impairment_RAW_VIEWING.tif' # SAVE TO SCRATCH

profile = cog_profiles.get('lzw')

profile.update(
    compress = 'DEFLATE',
    predictor = 3, # 2 for integer; 3 for float
    tiled = True,
    blockxsize = 512,
    blockysize = 512, 
    bigtiff = 'YES')

cog_translate(
    impairment_raster, # dont include argument
    cog_out_impairment_raw_viewing, # dont include argument
    profile, # dont include argument
    overview_resampling = 'bilinear', # Change based on data values
    nodata = -10,
    use_cog_driver = True,
    in_memory = False, 
    web_optimized = True)

Reading input: ./data/watershed_impairment/impairment_raw_rasterized.tif

Adding overviews...
Updating dataset tags...
Writing output to: /scratch/bfqp/cchan2/rasters/impairment_RAW_VIEWING.tif


In [13]:
# Validate cogeo raster
cog_validate(cog_out_impairment_raw_viewing)

(True, [], [])

In [14]:
# Copy from scratch back into projects folder
import shutil

cog_out_impairment_raw_viewing = '/scratch/bfqp/cchan2/rasters/impairment_RAW_VIEWING.tif'
cog_out_impairment_raw_viewing_PROJECTS = '/projects/bfqp/cchan2/data/watershed_impairment/impairment_RAW_VIEWING.tif'

shutil.copy(cog_out_impairment_raw_viewing, cog_out_impairment_raw_viewing_PROJECTS)

'/projects/bfqp/cchan2/data/watershed_impairment/impairment_RAW_VIEWING.tif'

In [15]:
# Check cogeo raster

cog_out_impairment_raw_viewing_PROJECTS = './data/watershed_impairment/impairment_RAW_VIEWING.tif'

with rasterio.open(cog_out_impairment_raw_viewing_PROJECTS, mode = 'r') as src:
    print(f'Profile: {src.profile}')
    print(f'Nodata: {src.nodata}')
    print(f'CRS: {src.crs}')
    print(f'Resolution {src.res}')
   
    # Check min and max rescaled values:
    
    # Set initial global min and max
    global_min = np.inf 
    global_max = -np.inf 
    
    # Set total_nodata
    total_nodata = 0

    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
        # Count nodata cells
        total_nodata += np.sum(data.mask)
        
    # Print the min and max rescaled values
    print(f'Min: {global_min}')
    print(f'Max: {global_max}')
    
    # % nodata cells
    total_cells = src.width * src.height
    percent_nodata = (total_nodata / total_cells) 
    print(f'% nodata cells: {percent_nodata}')

Profile: {'driver': 'GTiff', 'dtype': 'float32', 'nodata': -10.0, 'width': 182784, 'height': 107776, 'count': 1, 'crs': CRS.from_wkt('PROJCS["WGS 84 / Pseudo-Mercator",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Mercator_1SP"],PARAMETER["central_meridian",0],PARAMETER["scale_factor",1],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],EXTENSION["PROJ4","+proj=merc +a=6378137 +b=6378137 +lat_ts=0 +lon_0=0 +x_0=0 +y_0=0 +k=1 +units=m +nadgrids=@null +wktext +no_defs"],AUTHORITY["EPSG","3857"]]'), 'transform': Affine(38.2185141425881, 0.0, -14245416.087451734,
       0.0, -38.2185141425881, 6731350.458905771), 'blockxsize': 512, 'blockysize': 512, 'tiled': True, 'compress': '

In [ ]:
# Convert vector to raster but rasterize attribute values (pct_imp values) 

dist_gwt_REF = './data/SSURGO_raw/dist_GWT/gwt_inches.tif'
impairment_raster = './data/watershed_impairment/impairment_raw.tif'

# Create tuples of (geometry, value) pairs where the value is the attribute you want to "burn" into the raster
geom_value = ((geom, value) for geom, value in zip(impairment.geometry, impairment['pct_imp']))

# Open reference raster
ref_raster = rasterio.open(impervious_reprojected)

# Rasterize
rasterized_impairment = features.rasterize(geom_value,
                                           out_shape = ref_raster.shape,
                                           transform = ref_raster.transform,
                                           all_touched = False,
                                           fill = -99999, # Background value (nodata)
                                           merge_alg = MergeAlg.replace,
                                           dtype = rasterio.float32)

# Save
with rasterio.open(impairment_raster, mode = 'w',
                   driver = 'GTiff',
                   crs = ref_raster.crs,
                   transform = ref_raster.transform,
                   dtype = rasterio.float32,
                   count = 1,
                   width = ref_raster.width,
                   height = ref_raster.height,
                   nodata = -99999) as dst: 
        dst.write(rasterized_impairment.astype(rasterio.float32), indexes = 1)

# Close reference raster or else chaos
ref_raster.close()

print('Done!')

In [ ]:
# Apply min-max scaling - 8:49 AM, 8:54 AM global max, min - 9:12 finish

impairment_MATCH = './data/watershed_impairment/impairment_MATCH.tif'
impairment_standardized = './data/watershed_impairment/impairment_standardized.tif'

with rasterio.open(impairment_MATCH, mode = 'r') as src:
    profile = src.profile.copy()
    profile.update(compress='DEFLATE',
                   predictor = 3,
                   BIGTIFF = 'YES',
                   tiled = True, 
                   blockxsize = 128, 
                   blockysize = 128)
    
    # Set initial global min and max
    global_min = np.inf # Highest possible number so anythign will automatically be less
    global_max = -np.inf # Lowest possible number so anything will automatically be greater
    
    for ji, window in src.block_windows(1):
        data = src.read(1, window = window, masked = True)
        
        # Get local min and max for each window
        local_min = data.min()
        local_max = data.max()
        
        # Get global min and max by comparing the initial global min and max to each local min and max for all windows
        global_min = min(global_min, local_min) # Compares global_min to window min
        global_max = max(global_max, local_max) # Compared global_max to window max
        
    # Sanity check: make sure min and max are reasonable values
    print(f'Global min: {global_min}')
    print(f'Global max: {global_max}')
    
    with rasterio.open(impairment_standardized, mode = 'w', **profile) as dst:
        for ji, window in src.block_windows(1):
            data = src.read(1, window = window, masked = True)
            
            # Apply min-max scaling
            rescaled = ((data - global_min) / (global_max - global_min) * 10)
            
            # Fill masked (nodata values) with the nodata value
            rescaled_filled = rescaled.filled(src.nodata)
            
            # Write out raster
            dst.write(rescaled_filled.astype(np.float32), 1, window = window)